In [0]:
df=spark.read.format("csv").\
            options(header='true', inferSchema='true').\
            load('/Volumes/databricksdan/bronze/bronze_volume/customers/dim_customers.csv')

In [0]:
df.display(5)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=df.withColumn("name",upper(col("name")))

In [0]:
df.show(5)

In [0]:
df=df.withColumn("domain",split(col("email"),"@")[1])

In [0]:
df.show(5)

In [0]:
display(df.groupBy("domain").agg(count(col("customer_id")).alias("total_customer")).sort(col("total_customer").desc()))

Databricks visualization. Run in Databricks to view.

In [0]:
df=df.withColumn("processDate",current_timestamp())
display(df)


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricksdan.silver.customer_enr"):
    dlt_obj=DeltaTable.forName(spark,"databricksdan.silver.customer_enr")
    dlt_obj.alias('tgt').merge(df.alias('src'),'tgt.customer_id=src.customer_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df.write.format("delta").mode("append").saveAsTable("databricksdan.silver.customer_enr")

In [0]:
df_product= spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/databricksdan/bronze/bronze_volume/products")
display(df_product)


In [0]:
df_product=df_product.withColumn("processDate",current_timestamp())
display(df_product)


In [0]:
df_product.groupBy("category").agg(avg(col("price")).alias("avg_price")).show(5)

In [0]:
if spark.catalog.tableExists("databricksdan.silver.products_enr"):
    dlt_obj=DeltaTable.forName(spark,"databricksdan.silver.products_enr")
    dlt_obj.alias('tgt').merge(df.alias('src'),'tgt.product_id=src.product_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_product.write.format("delta").mode("append").saveAsTable("databricksdan.silver.productrs_enr")

In [0]:
df_stores= spark.read.format("csv").options(header=True,inferSchema=True).load("/Volumes/databricksdan/bronze/bronze_volume/stores")


In [0]:
display(df_stores)

In [0]:
df_stores=df_stores.withColumn("store_name",regexp_replace(col("store_name"),"_",""))

In [0]:
df_stores=df_stores.withColumn("processDate",current_timestamp())
display(df_stores)

In [0]:
if spark.catalog.tableExists("databricksdan.silver.stores_enr"):
    dlt_obj=DeltaTable.forName(spark,"databricksdan.silver.stores_enr")
    dlt_obj.alias('tgt').merge(df.alias('src'),'tgt.store_id=src.store_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_product.write.format("delta").mode("append").saveAsTable("databricksdan.silver.stores_enr")

In [0]:
df_sales=spark.read.format("csv").options(header=True,inferSchema=True).load("/Volumes/databricksdan/bronze/bronze_volume/sales")
display(df_sales)

In [0]:
df_sales=df_sales.withColumn("price_sale", round(col("total_amount")/col("quantity"),2))
df_sales=df_sales.withColumn("processDate",current_timestamp())
display(df_sales)

In [0]:
if spark.catalog.tableExists("databricksdan.silver.sales_enr"):
    dlt_obj=DeltaTable.forName(spark,"databricksdan.silver.sales_enr")
    dlt_obj.alias('tgt').merge(df.alias('src'),'tgt.sale_id=src.sale_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_product.write.format("delta").mode("append").saveAsTable("databricksdan.silver.sales_enr")